Random Forest V3A + n_estimators 200

Baut auf V2 Max Depth 10 auf

In [1]:
import os
import random
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mplconfig")
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("scikit-learn importiert!")

scikit-learn importiert!


In [2]:
# Konfiguration (RF V1)
DATA_ROOT = Path("mm-fit")
FS_TARGET = 50.0
DT_MS = int(round(1000.0 / FS_TARGET))  # 20 ms
WINDOW_SIZE = 250                        # 5 Sekunden @ 50Hz
STRIDE_SAMPLES = int(0.2 * FS_TARGET)   # 0.2 Sekunden -> 10 Samples

# Paper-Splits
TRAIN_IDS = [1, 2, 3, 4, 6, 7, 8, 16, 17, 18]
VAL_IDS = [14, 15, 19]
TEST_SEEN_IDS = [9, 10, 11]
TEST_UNSEEN_IDS = [0, 5, 12, 13, 20]

EXERCISE_ORDER = [
    "squats", "lunges", "bicep_curls", "situps", "pushups",
    "tricep_extensions", "dumbbell_rows", "jumping_jacks",
    "dumbbell_shoulder_press", "lateral_shoulder_raises", "non-exercise",
]

label_to_id = {name: i for i, name in enumerate(EXERCISE_ORDER)}
id_to_label = {i: name for name, i in label_to_id.items()}
N_CLASSES = len(EXERCISE_ORDER)

print("Konfiguration gesetzt. Klassen:", N_CLASSES)

Konfiguration gesetzt. Klassen: 11


In [3]:
def session_name(session_id: int) -> str:
    return f"w{session_id:02d}"


def load_session_raw(session_id: int):
    w = session_name(session_id)
    base = DATA_ROOT / w
    acc = np.load(base / f"{w}_sw_r_acc.npy")
    gyr = np.load(base / f"{w}_sw_r_gyr.npy")
    labels = pd.read_csv(
        base / f"{w}_labels.csv",
        header=None,
        names=["start", "end", "reps", "exercise"],
    )
    return acc, gyr, labels


def assign_labels_by_frame(frames, labels_df):
    y = np.full(frames.shape[0], label_to_id["non-exercise"], dtype=np.int32)
    for _, row in labels_df.iterrows():
        ex = row["exercise"]
        if ex not in label_to_id:
            continue
        mask = (frames >= int(row["start"])) & (frames <= int(row["end"]))
        y[mask] = label_to_id[ex]
    return y


def resample_sw_r_to_50hz(acc, gyr, labels_df):
    acc = acc[np.argsort(acc[:, 1])]
    gyr = gyr[np.argsort(gyr[:, 1])]
    acc_ts = acc[:, 1].astype(np.float64)
    gyr_ts = gyr[:, 1].astype(np.float64)
    t_start = max(acc_ts.min(), gyr_ts.min())
    t_end = min(acc_ts.max(), gyr_ts.max())
    new_ts = np.arange(t_start, t_end + 1, DT_MS, dtype=np.float64)

    acc_xyz = np.stack([
        np.interp(new_ts, acc_ts, acc[:, 2]),
        np.interp(new_ts, acc_ts, acc[:, 3]),
        np.interp(new_ts, acc_ts, acc[:, 4]),
    ], axis=1)
    gyr_xyz = np.stack([
        np.interp(new_ts, gyr_ts, gyr[:, 2]),
        np.interp(new_ts, gyr_ts, gyr[:, 3]),
        np.interp(new_ts, gyr_ts, gyr[:, 4]),
    ], axis=1)

    acc_frames = acc[:, 0].astype(np.int64)
    acc_labels = assign_labels_by_frame(acc_frames, labels_df)
    nearest_idx = np.searchsorted(acc_ts, new_ts, side="left")
    nearest_idx = np.clip(nearest_idx, 0, len(acc_ts) - 1)
    prev_idx = np.clip(nearest_idx - 1, 0, len(acc_ts) - 1)
    choose_prev = np.abs(acc_ts[prev_idx] - new_ts) < np.abs(acc_ts[nearest_idx] - new_ts)
    nn_idx = np.where(choose_prev, prev_idx, nearest_idx)
    y_resampled = acc_labels[nn_idx]

    x_resampled = np.concatenate([acc_xyz, gyr_xyz], axis=1).astype(np.float32)
    return x_resampled, y_resampled.astype(np.int32)


# Alle Sessions laden
session_data = {}
for sid in range(21):
    acc, gyr, labels = load_session_raw(sid)
    x, y = resample_sw_r_to_50hz(acc, gyr, labels)
    session_data[sid] = {"x": x, "y": y}

print("Fertig. Sessions geladen:", len(session_data))

Fertig. Sessions geladen: 21


In [4]:
def fit_standardizer(train_ids):
    x_all = np.concatenate([session_data[sid]["x"] for sid in train_ids], axis=0)
    mean = x_all.mean(axis=0)
    std = x_all.std(axis=0) + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)

mean_vec, std_vec = fit_standardizer(TRAIN_IDS)

for sid in range(21):
    session_data[sid]["x"] = (session_data[sid]["x"] - mean_vec) / std_vec

print("Normalisierung gesetzt.")
print("Mean:", np.round(mean_vec, 4))
print("Std:", np.round(std_vec, 4))

Normalisierung gesetzt.
Mean: [ 4.2045 -3.3051  1.9853 -0.0049 -0.0077 -0.005 ]
Std: [6.3585 4.8538 4.3254 1.0802 0.8402 0.9931]


In [5]:
def extract_features(window):
    """
    window: numpy array (250, 6)
    gibt 30 Features zurück: 5 Features × 6 Kanäle
    """
    features = []
    for ch in range(6):
        signal = window[:, ch]
        features.append(np.mean(signal))           # Mean
        features.append(np.std(signal))            # Standard Deviation
        features.append(np.min(signal))            # Min
        features.append(np.max(signal))            # Max
        features.append(np.sqrt(np.mean(signal**2)))  # RMS
    return np.array(features, dtype=np.float32)


# Test: eine einzelne Fenster ausprobieren
test_window = session_data[0]["x"][:250]
test_features = extract_features(test_window)
print("Feature Shape:", test_features.shape)  # sollte (30,) sein
print("Erste 10 Features:", np.round(test_features[:10], 4))

Feature Shape: (30,)
Erste 10 Features: [-1.6790e+00  6.4710e-01 -3.1050e+00 -5.7570e-01  1.7994e+00  1.1000e-03
  4.1410e-01 -1.0923e+00  1.1476e+00  4.1410e-01]


In [6]:
def build_feature_matrix(split_ids):
    """
    Erstellt Feature-Matrix X und Label-Vektor y für einen Split.
    """
    X_list = []
    y_list = []

    for sid in split_ids:
        x = session_data[sid]["x"]
        y = session_data[sid]["y"]
        max_start = len(y) - WINDOW_SIZE

        if max_start < 0:
            continue

        starts = np.arange(0, max_start + 1, STRIDE_SAMPLES)

        for st in starts:
            window = x[st:st + WINDOW_SIZE]
            label = np.bincount(y[st:st + WINDOW_SIZE], minlength=N_CLASSES).argmax()
            features = extract_features(window)
            X_list.append(features)
            y_list.append(label)

    return np.array(X_list), np.array(y_list)


print("Baue Train-Split...")
X_train, y_train = build_feature_matrix(TRAIN_IDS)
print("Baue Val-Split...")
X_val, y_val = build_feature_matrix(VAL_IDS)
print("Baue Test Seen-Split...")
X_seen, y_seen = build_feature_matrix(TEST_SEEN_IDS)
print("Baue Test Unseen-Split...")
X_unseen, y_unseen = build_feature_matrix(TEST_UNSEEN_IDS)

print(f"\nX_train Shape: {X_train.shape}")
print(f"X_unseen Shape: {X_unseen.shape}")

Baue Train-Split...
Baue Val-Split...
Baue Test Seen-Split...
Baue Test Unseen-Split...

X_train Shape: (119123, 30)
X_unseen Shape: (60128, 30)


In [7]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)

print("Training startet...")
rf.fit(X_train, y_train)
print("Training fertig!")

Training startet...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    8.0s


Training fertig!


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    8.8s finished


In [8]:
def evaluate_rf(name, X, y_true):
    y_pred = rf.predict(X)
    acc = (y_true == y_pred).mean()
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n[{name}] Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    print(classification_report(
        y_true, y_pred,
        target_names=[id_to_label[i] for i in range(N_CLASSES)],
        digits=4
    ))
    return y_pred

y_seen_pred = evaluate_rf("Test Seen", X_seen, y_seen)
y_unseen_pred = evaluate_rf("Test Unseen", X_unseen, y_unseen)


[Test Seen] Accuracy: 0.9621 | Macro-F1: 0.9162
                         precision    recall  f1-score   support

                 squats     0.7947    1.0000    0.8856      1026
                 lunges     0.8343    0.9896    0.9053      1343
            bicep_curls     0.9359    0.9832    0.9589       950
                 situps     0.8602    1.0000    0.9248      1341
                pushups     0.7974    0.9938    0.8848       808
      tricep_extensions     0.9320    1.0000    0.9648      1082
          dumbbell_rows     0.9265    0.9557    0.9408       857
          jumping_jacks     0.8457    0.5781    0.6867       474
dumbbell_shoulder_press     0.9588    0.9950    0.9766      1194
lateral_shoulder_raises     0.9613    0.9891    0.9750      1105
           non-exercise     0.9915    0.9590    0.9750     30861

               accuracy                         0.9621     41041
              macro avg     0.8944    0.9494    0.9162     41041
           weighted avg     0.9657    0

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 200 out of 200 | elapsed:    0.1s finished
